In [ ]:
import sys
!{sys.executable} -m pip install xgboost

In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)

In [ ]:
df1 = pd.read_csv('product_type_1.csv',header=[0,1])

In [ ]:
df1

In [ ]:
df1.info()

In [ ]:
# 2) 간단한 EDA
print("\n결측치:\n", df1.isnull().sum().sort_values(ascending=False).head(10)) 
print("\n타겟 비율:\n", df1[('defect_flag', 'is_defect')].value_counts(normalize=True))

In [ ]:
# y: 불량 여부(0/1)
y = df1[('defect_flag', 'is_defect')].astype(int)

 # # y: Defects 그룹 전체 (멀티라벨)
# y_cols = [c for c in df.columns if c[0] == 'Defects']
# y = df[y_cols].astype(int)

X = df1[['process','sensor']].copy()
X = X.drop(columns=[('process','id'), ('process','product_type')])
X


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
print("train:", X_train.shape)
print("test:", X_test.shape)

print("train 불량률:", y_train.mean())
print("test 불량률:", y_test.mean())

In [ ]:
X_train.isnull().sum()

In [ ]:
# y_train의 분포 확인
sns.countplot(x=y_train)
plt.title('Distribution of Defects (0: Normal, 1: Defect)')
plt.show()

In [ ]:
X_train['process','shot'].value_counts()

In [ ]:
X_train.info()

In [ ]:
# IQR 기반 이상치 탐지 함수 정의
def get_outlier_stats(df):
    outlier_list = []
    
    for col in X_train.columns:
        # 숫자형 데이터만 처리
        if pd.api.types.is_numeric_dtype(df[col]):
            Q1 = X_train[col].quantile(0.25)
            Q3 = X_train[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # 이상치 개수 계산
            outliers = X_train[(X_train[col] < lower_bound) | (X_train[col] > upper_bound)]
            outlier_count = len(outliers)
            outlier_ratio = (outlier_count / len(X_train)) * 100
            
            outlier_list.append({
                'Column': col,
                'Q1': Q1,
                'Q3': Q3,
                'IQR': IQR,
                'Lower Bound': lower_bound,
                'Upper Bound': upper_bound,
                'Outlier Count': outlier_count,
                'Outlier Ratio (%)': round(outlier_ratio, 2)
            })
            
    return pd.DataFrame(outlier_list)


outlier_summary = get_outlier_stats(X_train)

# 이상치가 많은 순서대로 정렬
display(outlier_summary.sort_values(by='Outlier Count', ascending=False))


# process 관련 주요 수치들의 분포 확인
process_cols = [col for col in X_train.columns if 'process' in str(col)]
plt.figure(figsize=(15, 8))
X_train[process_cols].boxplot()
plt.xticks(rotation=90)
plt.title("Outliers in Process Data")
plt.show()


# sensor 관련 주요 수치들의 분포 확인
sensor_cols = [col for col in X_train.columns if 'sensor' in str(col)]
plt.figure(figsize=(15, 8))
X_train[sensor_cols].boxplot()
plt.xticks(rotation=90)
plt.title("Outliers in Sensor Data")
plt.show()

In [ ]:
# Process 관련 컬럼 히스토그램
process_cols = [col for col in X_train.columns if 'process' in str(col) and 'shot' not in str(col)]

plt.figure(figsize=(20, 15))
plt.suptitle("공정(Process) 데이터 분포 및 이상치 확인", fontsize=20)

for i, col in enumerate(process_cols, 1):
    plt.subplot(4, 4, i)
    sns.histplot(X_train[col], kde=True, color='skyblue')
    plt.title(f'Dist: {col[1]}') # MultiIndex의 하위 컬럼명 사용
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

# Sensor 관련 컬럼 히스토그램
sensor_cols = [col for col in X_train.columns if 'sensor' in str(col)]

plt.figure(figsize=(20, 15))
plt.suptitle("센서(Sensor) 데이터 분포 및 이상치 확인", fontsize=20)

for i, col in enumerate(sensor_cols, 1):
    plt.subplot(4, 4, i)
    sns.histplot(X_train[col], kde=True, color='salmon')
    plt.title(f'Dist: {col[1]}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

In [ ]:
defect_cols = [col for col in df1.columns if 'defect' in str(col)]

for col in defect_cols:
    print(f"--- [Column: {col}] ---")
    print(df1[col].value_counts())
    print("\n") 

# 각 컬럼의 value_counts를 데이터프레임 하나로 병합
counts_summary = df1[defect_cols].apply(lambda x: x.value_counts()).fillna(0).astype(int)
display(counts_summary)

In [ ]:
# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (df1[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

In [ ]:
d = df1.copy()
d_reduced = d.drop(columns=only_zero_cols)

d_reduced

In [ ]:
defect_cols = [col for col in d_reduced.columns if 'defect' in str(col)]

# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (d_reduced[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

In [ ]:
df2 = pd.read_csv('product_type_2.csv',header=[0,1])

In [ ]:
defect_cols = [col for col in df2.columns if 'defect' in str(col)]

for col in defect_cols:
    print(f"--- [Column: {col}] ---")
    print(df2[col].value_counts())
    print("\n") 

# 각 컬럼의 value_counts를 데이터프레임 하나로 병합
counts_summary = df2[defect_cols].apply(lambda x: x.value_counts()).fillna(0).astype(int)
display(counts_summary)

In [ ]:
# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (df2[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

In [ ]:
dd = df2.copy()
d_reduced = dd.drop(columns=only_zero_cols)

d_reduced

In [ ]:
defect_cols = [col for col in d_reduced.columns if 'defect' in str(col)]

# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (d_reduced[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

In [ ]:
TARGET_GROUPS = ["process"]

# 보통 제외하는 컬럼 (식별자/순번)
EXCLUDE_SUBCOLS = {"id", "shot"}

target_cols = [
    c for c in df1.columns
    if c[0] in TARGET_GROUPS and c[1] not in EXCLUDE_SUBCOLS
]


# 숫자형으로 강제 변환(문자 섞인 컬럼 대비)
df_num = df1[target_cols].apply(pd.to_numeric, errors="coerce")

print("분석 대상 컬럼 수:", df_num.shape[1])

# 1) IQR 기반 이상치 탐지
Q1 = df_num.quantile(0.25)
Q3 = df_num.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

is_out_iqr = (df_num.lt(lower) | df_num.gt(upper))  # DataFrame[bool]

iqr_outlier_count = is_out_iqr.sum().sort_values(ascending=False)
iqr_outlier_rate = (is_out_iqr.mean() * 100).sort_values(ascending=False)

iqr_summary = pd.DataFrame({
    "이상치 갯수": iqr_outlier_count,
    "이상치 비율(%)": iqr_outlier_rate.round(3),
    "하한선": lower,
    "상한선": upper,
    "Q1(25%)": Q1,
    "Q3(75%)": Q3,
    "IQR": IQR
}).sort_values("이상치 비율(%)", ascending=False)

print("\n[IQR 이상치 TOP 15]")
display(iqr_summary.head(15))

In [ ]:
# 이상치가 있는 컬럼만 (이상치 갯수 > 0)
top11_cols = iqr_summary[iqr_summary["이상치 갯수"] > 0].head(11).index

for col in top11_cols:
    lower_val = lower[col]
    upper_val = upper[col]

    s = df_num[col].dropna()   # ✅ df가 아니라 df_num 사용 (숫자형 보장)
    mean_val = s.mean()

    overall_min = s.min()
    overall_max = s.max()

    # 이상치 마스크/개수
    low_mask  = s < lower_val
    high_mask = s > upper_val
    out_cnt = int((low_mask | high_mask).sum())

    # ✅ 평균 대비 % 차이 (mean이 0이면 0으로 나누기 방지)
    if mean_val == 0 or np.isclose(mean_val, 0):
        min_pct = np.nan
        max_pct = np.nan
    else:
        min_pct = (overall_min - mean_val) / mean_val * 100
        max_pct = (overall_max - mean_val) / mean_val * 100

    print("="*60)
    print(col)  # MultiIndex 그대로 출력 (원하면 아래에서 col[1]만 출력도 가능)
    print(f"IQR 경계: lower={lower_val:.6g}, upper={upper_val:.6g} | 이상치 수={out_cnt}")

    # ✅ min — mean — max
    print(f"전체 범위: min={overall_min:.6g} | mean={mean_val:.6g} | max={overall_max:.6g}")

    # ✅ mean 대비 % 차이
    print(f"평균 대비: min {min_pct:+.2f}% | max {max_pct:+.2f}%")

In [ ]:
# 이상치 있는 컬럼 TOP11
top11_cols = iqr_summary[iqr_summary["이상치 갯수"] > 0].head(11).index

rows = []
for col in top11_cols:
    s = df_num[col].dropna()

    lower_val = lower[col]
    upper_val = upper[col]

    mean_val = s.mean()
    min_val  = s.min()
    max_val  = s.max()

    # 이상치 수/비율 (df_num 기준)
    out_mask = (s < lower_val) | (s > upper_val)
    out_cnt = int(out_mask.sum())
    out_rate = out_cnt / len(s) * 100 if len(s) else np.nan

    # 평균 대비 % 차이
    if mean_val == 0 or np.isclose(mean_val, 0):
        min_pct = np.nan
        max_pct = np.nan
    else:
        min_pct = (min_val - mean_val) / mean_val * 100
        max_pct = (max_val - mean_val) / mean_val * 100

    # 컬럼명 깔끔하게(멀티인덱스면 2번째 레벨 사용)
    name = col[1] if isinstance(col, tuple) and len(col) > 1 else str(col)

    rows.append({
        "col": name,
        "min": min_val,
        "mean": mean_val,
        "max": max_val,
        "min_vs_mean_%": min_pct,
        "max_vs_mean_%": max_pct,
        "outlier_cnt": out_cnt,
        "outlier_rate_%": out_rate,
        "IQR": float(IQR[col])
    })

viz_df = pd.DataFrame(rows).sort_values("outlier_rate_%", ascending=False).reset_index(drop=True)
display(viz_df)

In [ ]:
top11 = iqr_summary.sort_values("이상치 비율(%)", ascending=False).head(11)

labels = [col[1] for col in top11.index]   # 컬럼 이름
values = top11["이상치 비율(%)"]

plt.figure(figsize=(10,6))

plt.bar(labels, values)

plt.xticks(rotation=45)
plt.ylabel("Outlier Rate (%)")
plt.title("Top 11 Columns with Highest Outlier Rates")

plt.tight_layout()
plt.show()

In [ ]:
top11_cols = iqr_summary.sort_values("이상치 비율(%)", ascending=False).head(11).index

for col in top11_cols:

    s = df_num[col].dropna()
    
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False

    plt.figure(figsize=(6,4))

    # x값 점 간격
    x = np.random.normal(0, 0.04, size=len(s))

    plt.scatter(x, s, alpha=0.3, s=8)

    # 평균 표시
    mean_val = s.mean()
    plt.axhline(mean_val, linestyle="--")

    plt.title(f"{col[1]} 이상치 분포도")
    plt.ylabel("Value")
    plt.xticks([])

    plt.tight_layout()
    plt.show()